In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib
matplotlib.rcParams.update({ "pgf.texsystem": "pdflatex", 'font.family': 'serif', 'pgf.rcfonts': False,
                            "savefig.transparent" : True,  "svg.fonttype": 'none',  })
import matplotlib.pyplot as plt
plt.rc('font', family='serif')
plt.rc('text', usetex=False)
plt.rc("svg", fonttype='none')
plt.rc('text.latex', preamble=
       r'\usepackage{amsmath}'\
       + "\n" + r'\usepackage{amssymb}'
       )
import matplotlib.figure, matplotlib.axes
import sys, os
print(sys.path)
thisnotebookdir=os.path.abspath("")
WORKSPACE=os.path.abspath(os.path.join(thisnotebookdir, "..", "..", ".."))
sys.path.insert(0, os.path.normpath(os.path.join(WORKSPACE, "deepracing", "deepracing_py")))
sys.path.insert(0, os.path.normpath(os.path.join(WORKSPACE, "deepracing", "DCNN-Pytorch")))
sys.path.insert(0, os.path.normpath(os.path.join(WORKSPACE, "deepracing_ros", "deepracing_rclpy")))
# sys.path.insert(0, os.path.normpath(os.path.join(ROOTDIR, "..", "..", "..", "..", "build", "deepracing_rclpy")))
print(sys.path)

import numpy as np
import pickle as pkl
import tqdm
import collections.abc
import torch
from utils import PredictionResults
import matplotlib.transforms
import matplotlib.artist
import matplotlib.collections
import matplotlib.lines
import matplotlib.axes
import matplotlib.figure
import matplotlib.scale
import matplotlib.patches
from matplotlib.gridspec import GridSpec
from scipy.spatial.transform import Rotation
import yaml 
import torch.utils.data as torchdata

from deepracing_models.data_loading import SubsetFlag
import deepracing_models.math_utils as mu
import deepracing_models.math_utils.dynamics as dynamics
import deepracing_models.data_loading.file_datasets as FD
import deepracing_models.data_loading.utils.file_utils as file_utils
import deepracing

In [ ]:
import scipy.interpolate
trackname="Monza_cavsim"

trackmap=deepracing.searchForTrackmap(trackname, [  os.path.join(WORKSPACE, "cavauto", "dockerimages", "cavsim", "deepracing_trackmaps"),
                                                    os.path.join(WORKSPACE, "deepracing_ros", "deepracing_launch", "maps"),
                                                  ], align=True)

rlpoints = np.concatenate([trackmap.raceline[k] for k in ["x","y"]], axis=1)
rlspeeds = trackmap.raceline["speed"][:,0]
rltimes = trackmap.raceline["time"][:,0]

rlspline = scipy.interpolate.make_interp_spline(rltimes, rlpoints, k=2, bc_type="periodic")



In [ ]:
trand = np.random.choice(rltimes, size=1).item()
# trand=29.717199325561523
# trand=8.100700378417969
# trand=60.15019989013672
# trand=3.500699996948242
trand=61.58300018310547
print(trand)


T_F = 8.0

tsamp = torch.linspace(trand, trand+T_F, steps=200, dtype=torch.float64)%rltimes[-1]

Psamp = torch.as_tensor(rlspline(tsamp)).type_as(tsamp)
Vsamp = torch.as_tensor(rlspline(tsamp, nu=1)).type_as(tsamp)
Speedsamp = torch.linalg.norm(Vsamp, dim=1, ord=2.0)
Tausamp = Vsamp/Speedsamp[:,None]

Nusamp = Tausamp[:,[1,0]].clone()
Nusamp[:,0]*=-1.0
columnwidth=3.5
figsize = 2.0*np.asarray([columnwidth, columnwidth*0.875])
figname="example_theta"
plt.close(fig=figname)
fig, ax = plt.subplots(figsize=figsize, num=figname, frameon=False)
fig.set_frameon(False)
ax.set_frame_on(False)
for k in ax.spines.keys():
    ax.spines[k].set_visible(False)
ax.set_xticks([])
ax.set_yticks([])

x0ego = Psamp[0] + 15.6*Nusamp[0]
v0ego = Vsamp[0]

xend = Psamp[-1]
vend = Vsamp[-1]

matrix_factory = mu.BezierMatrixFactory(3)
control_points, tswitch = mu.compositeBezierFit(tsamp, Psamp, 3, Y_0=x0ego, dYdT_0=v0ego, Y_f=xend, dYdT_f=vend, kbezier=matrix_factory.comb_factors.shape[0]-1)
tstart = tswitch[:-1]
tend = tswitch[1:]
deltaT = tend - tstart

curve_eval, _ = mu.compositeBezierEval(tstart, deltaT, control_points, tsamp, matrix_factory)
control_points_plot = control_points[:,:-1].reshape(-1,control_points.shape[-1])
rlartist, = ax.plot(Psamp[:,0], Psamp[:,1], linewidth=1, color="green")
ax.scatter(control_points_plot[:,0], control_points_plot[:,1], color="blue", s=2**1.0)
ax.plot(curve_eval[:,0], curve_eval[:,1], color = "blue", linewidth = rlartist.get_linewidth())
plotdir="/u/ttw2xk/cavws/plots"

fig.savefig(os.path.join(plotdir, "example_t_theta.svg"), bbox_inches="tight", pad_inches=0.0)
# rlartist.remove()


In [ ]:

figname="example_cbc"
plt.close(fig=figname)
fig, ax = plt.subplots(figsize=figsize, num=figname, frameon=False)
fig.set_frameon(False)
ax.set_frame_on(False)
for k in ax.spines.keys():
    ax.spines[k].set_visible(False)
ax.set_xticks([])
ax.set_yticks([])
ax.scatter(control_points_plot[:,0], control_points_plot[:,1], color="blue", s=2**1.0)
for idx in range(tstart.shape[0]):
    idxcurrsegment=(tsamp>=tstart[idx])*(tsamp<tend[idx])
    ax.plot(curve_eval[idxcurrsegment,0], curve_eval[idxcurrsegment,1], linewidth=1)
fig.savefig(os.path.join(plotdir, "example_cbc.svg"), bbox_inches="tight", pad_inches=0.0)